In [ ]:
#****حفظ الموديل مع استخدام Hsv
import os
import numpy as np
from sklearn.model_selection import KFold
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Set data paths
data_dir = r'D:\all\AI\photo\editequlization\leaf_output3\train'  # Path to training data
test_dir = r'D:\all\AI\photo\editequlization\leaf_output3\Test'   # Path to testing data

# Set image dimensions and batch size
img_height, img_width = 64, 64  # Reducing dimensions to save memory
batch_size = 16  # Reducing batch size
num_classes = 3  # Number of classes

# Initialize ImageDataGenerator for training data
data_gen = ImageDataGenerator(rescale=1./255)  # Data generation with rescaling

# Set up training data generator
train_generator = data_gen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
    class_mode='categorical'
)

# Define the CNN model
def create_model():
    model = Sequential()
    model.add(Input(shape=(img_height, img_width, 3)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(256, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Set up cross-validation
# x=5
num_folds = 5
fold_accuracies = []

kf = KFold(n_splits=num_folds, shuffle=True)

# Train and evaluate using the data generator
for fold, (train_index, val_index) in enumerate(kf.split(train_generator)):
    print(f"Training fold {fold + 1}/{num_folds}...")

    # Reset the data generator for each fold
    train_generator.reset()
    
    model = create_model()

    # Train the model using the generator
    history = model.fit(train_generator,
                        steps_per_epoch=len(train_generator),
                        epochs=25)

    # Evaluate the model on the test data
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='categorical'
    )

    test_loss, test_acc = model.evaluate(test_generator)
    fold_accuracies.append(test_acc)
    print(f"Fold {fold + 1} accuracy: {test_acc}")

    # Save the model if it's the last fold
    if fold == num_folds - 1:
        model.save("final_model.h5")
        print("Final model saved as 'final_model.h5'")

# Print the mean accuracy across all folds
print(f'Mean accuracy across folds: {sum(fold_accuracies) / num_folds}')


test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False 
)

test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.4f}")
